# Phase 4h: the Fang et al. (2025) test

**Status:** Part A (data) done 2026-11-04. Part B (models) not run yet; this notebook is extended then.

**What Fang did:** Fang, Lee & Li (2025) regress renewable energy consumption on AI job postings, policy uncertainty and controls, all in logs, with state fixed effects, for 50 states + DC in 2018-2021 (N = 204). Their AI coefficient runs from 0.0396 to 0.0561.

**How this differs from the Zhang test (4b-4e):** different outcome (all renewable energy use, not the electricity share), different AI measure (job postings, not data-center jobs), different controls, no year effects and no lag. The Zhang test stays as it is.

In [1]:
import pandas as pd
from IPython.display import display
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40); pd.set_option('display.max_colwidth', 70); pd.set_option('display.max_rows', 100)
L = '../logs/2026-11-04/'
post = pd.read_csv('../data/ai_job_postings_state.csv')
print('Postings:', post.shape, '| Fang panel:', pd.read_csv('../data/fang_panel.csv').shape)

Postings: (408, 13) | Fang panel: (561, 41)


## Coverage

Number of areas with data, out of 51, by variable and year.

In [2]:
display(pd.read_csv(L + 'fang_coverage.csv'))

,variable,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,first,last
0,re_bbtu,51,51,51,51,51,51,51,51,51,51,0,2015,2024
1,wind_bbtu,51,51,51,51,51,51,51,51,51,51,0,2015,2024
2,solar_bbtu,51,51,51,51,51,51,51,51,51,51,0,2015,2024
3,geothermal_bbtu,51,51,51,51,51,51,51,51,51,51,0,2015,2024
4,hydro_bbtu,51,51,51,51,51,51,51,51,51,51,0,2015,2024
5,biomass_bbtu,51,51,51,51,51,51,51,51,51,51,0,2015,2024
6,total_energy_bbtu,51,51,51,51,51,51,51,51,51,51,0,2015,2024
7,ep_tetcd,51,51,51,51,51,51,51,51,51,51,0,2015,2024
8,edu_bach_per1000,51,51,51,51,51,51,51,51,51,51,0,2015,2024
9,gdp_real_musd2017,51,51,51,51,51,51,51,51,51,51,51,2015,2025


## AI job postings: where each year comes from

In [3]:
t = post.groupby(['year', 'edition', 'vendor', 'vintage', 'figure'], dropna=False).agg(sum_of_51=('postings', 'sum'), page=('page', 'first')).reset_index()
t['national_change_pct'] = t.sum_of_51.pct_change().mul(100).round(0)
display(t)

,year,edition,vendor,vintage,figure,sum_of_51,page,national_change_pct
0,2018,2019,Burning Glass,ed2019_bg,Fig. 4.1.8 shows a scatter; values from its linked sheet,189030.0,82.0,NaN
1,2019,2022,Emsi Burning Glass,ed2022_ebg,"Public-data sheet, tab AI Job Postings by State; not printed in th...",325702.0,NaN,72.0
2,2020,2022,Emsi Burning Glass,ed2022_ebg,"Public-data sheet, tab AI Job Postings by State; not printed in th...",296937.0,NaN,-9.0
3,2021,2022,Emsi Burning Glass,ed2022_ebg,Fig. 4.1.6,404074.0,9.0,36.0
4,2022,2023,Lightcast,ed2023_lc,Fig. 4.1.6; share Fig. 4.1.8 (p. 11),795624.0,10.0,97.0
5,2023,2024,Lightcast,ed2024_lc,Fig. 4.2.7; share Fig. 4.2.9 (p. 18),461225.0,17.0,-42.0
6,2024,2025,Lightcast,ed2025_lc,Fig. 4.2.8; share Fig. 4.2.10 (p. 17),658548.0,16.0,43.0
7,2025,2026,Lightcast,ed2026_lc,Fig. 4.4.10; share Fig. 4.4.11 (p. 40),994372.0,40.0,51.0


## How to read the postings series

**In the postings extension (2019-2024), almost every year comes from a different edition. So a state's year-to-year change mixes real change with method changes.** Year effects remove shifts that hit all states proportionally, and the share-of-national row tests that. Anything uneven stays in as noise. Data-center jobs (BLS, one consistent series) remain the more reliable extension measure.

National totals swing +97% in 2022 and -42% in 2023 around reported method changes. Those are the shifts year effects can absorb, but only if they hit states proportionally.

## Checks on the postings table

In [4]:
display(pd.read_csv(L + 'ai_postings_checks.csv'))

,year,check,result,detail
0,2018,51 rows,ok,"sum of 51 = 189,030; yoy"
1,2019,51 rows,ok,"sum of 51 = 325,702; yoy +72%"
2,2020,51 rows,ok,"sum of 51 = 296,937; yoy -9%"
3,2021,51 rows,ok,"sum of 51 = 404,074; yoy +36%"
4,2022,51 rows,ok,"sum of 51 = 795,624; yoy +97%"
5,2023,51 rows,ok,"sum of 51 = 461,225; yoy -42%"
6,2024,51 rows,ok,"sum of 51 = 658,548; yoy +43%"
7,2025,51 rows,ok,"sum of 51 = 994,372; yoy +51%"
8,2022,sum vs national total implied by published shares,ok,"implied 795,471; sum 795,624; gap -0.02%; max |published - derived..."
9,2023,sum vs national total implied by published shares,ok,"implied 461,332; sum 461,225; gap +0.02%; max |published - derived..."


## Edition overlap

Levels can't be compared when one edition covers Jan-Sep and the other the full year, but each state's **share of U.S. AI postings** can. These are the only years two editions report for all states.

**Reading it:** the state pattern is very similar across editions, but high correlations are driven by the ordering of large and small states, so they don't clear within-state noise. For 2019 the median state's share still differs by 14.8% (SD of the log ratio 0.162), part of which is Jan-Sep against the full year. For 2020, with both full years, the median gap is 0.7%, so a one-year revision within Burning Glass is small.

In [5]:
display(pd.read_csv(L + 'ai_postings_overlap.csv').drop(columns=['comparison']))

,year,edition_a,edition_b,n_states,pearson_share,spearman_share,pearson_ln_share,level_ratio_b_over_a,sd_ln_share_ratio,median_abs_pct_gap,largest_gaps
0,2019,"2019 edition, Jan-Sep","2022 edition, full year",51,0.9978,0.9954,0.9969,2.0000,0.1616,14.7934,MT -45%; AK -45%; MS -41%
1,2020,2021 edition,2022 edition,51,1.0000,0.9996,1.0000,0.9883,0.0116,0.6547,WV +4%; OR +3%; VT +3%
2,2021,"2022 edition, PDF map","2022 edition, sheet",51,1.0000,1.0000,1.0000,1.0000,0.0000,0.0000,AK +0%; PA +0%; ND +0%


### Four states, across editions

The 2023-2026 editions print a share history for CA, NY, TX and WA, and it's the same in all four. For 2022 onward it matches our table. For 2018-2021 it's Lightcast's restated history, which differs from the Burning Glass tables we use. **WA's fall from 4.8% to 3.9% at 2021-2022 in our table doesn't exist in Lightcast's own series (3.94 to 3.93): it's a method change.**

In [6]:
display(pd.read_csv(L + 'ai_postings_select_states.csv'))

,state,year,series_2023,series_2024,series_2025,series_2026,table_share_pct,table_edition,alt_share_pct
0,CA,2018,22.522,22.52,22.52,22.522,25.959,2019,NaN
1,CA,2019,24.619,24.62,24.62,24.619,24.512,2022,27.458
2,CA,2020,20.877,20.88,20.88,20.877,21.055,2022,21.113
3,CA,2021,19.507,19.51,19.51,19.507,19.857,2022,NaN
4,CA,2022,17.867,17.87,17.87,17.867,17.870,2023,NaN
5,CA,2023,NaN,15.31,15.31,15.314,15.310,2024,NaN
6,CA,2024,NaN,NaN,15.70,15.697,15.700,2025,NaN
7,CA,2025,NaN,NaN,NaN,17.185,17.180,2026,NaN
8,NY,2018,7.037,7.04,7.04,7.037,8.221,2019,NaN
9,NY,2019,7.686,7.69,7.69,7.686,7.573,2022,8.811


### How much state shares move each year

SD across states of the year-to-year change in ln(state share). Postings shares move 2 to 3 times as much as data-center jobs in nearly every step, including the two same-edition steps. Steps across editions are somewhat larger on average (0.227 against 0.195). So postings are noisy at the state level for more reasons than edition changes.

In [7]:
display(pd.read_csv(L + 'ai_postings_share_changes.csv'))

,step,editions,same_edition,national_change_pct,sd_ln_share_change_postings,median_abs_ln_share_change,sd_ln_share_change_dc_jobs,n_dc_jobs,largest
0,2018->2019,2019->2022,False,72.3018,0.2230,0.1395,0.1196,51,MS +90%; IN +81%; AK +81%
1,2019->2020,2022->2022,True,-8.8317,0.1738,0.1155,0.1294,51,HI +111%; ID +63%; ND +33%
2,2020->2021,2022->2022,True,36.0807,0.2166,0.1761,0.0957,51,AR +95%; WY +82%; ID +70%
3,2021->2022,2022->2023,False,96.9006,0.2663,0.1313,0.1045,51,AR +139%; OK +99%; KS +90%
4,2022->2023,2023->2024,False,-42.0298,0.2493,0.1400,0.0829,51,AK +133%; MT +85%; ND +82%
5,2023->2024,2024->2025,False,42.7824,0.2004,0.1217,0.0618,51,MT +93%; WV +70%; MD -36%
6,2024->2025,2025->2026,False,50.9946,0.1951,0.0905,0.0775,47,VT +126%; WY +78%; WV +69%


## Part B

Not run yet: replication, bridge, moderation, by type, heterogeneity, placebo and robustness.